In [53]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import sympy as sp
from IPython.display import display, Math

In [54]:
tickers = ["KO", "PEP"]

prices = pd.read_csv("prices_KO_PEP.csv", index_col="Date", parse_dates=True)

prices.head()

,KO,PEP
Date,,
2021-01-04,44.527077,119.999931
2021-01-05,44.037586,120.357574
2021-01-06,42.636620,118.885345
2021-01-07,42.164017,118.502716
2021-01-08,43.109238,119.925064


In [55]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1255 entries, 2021-01-04 to 2025-12-31
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   KO      1255 non-null   float64
 1   PEP     1255 non-null   float64
dtypes: float64(2)
memory usage: 29.4 KB


In [56]:
prices.isna().sum()

KO     0
PEP    0
dtype: int64

In [57]:
returns =\
(
    prices
    .pct_change()
    .dropna()
)

returns.head()

,KO,PEP
Date,,
2021-01-05,-0.010993,0.002980
2021-01-06,-0.031813,-0.012232
2021-01-07,-0.011084,-0.003218
2021-01-08,0.022418,0.012003
2021-01-11,-0.017228,-0.014496


In [58]:
returns.describe()

,KO,PEP
count,1254.000000,1254.000000
mean,0.000394,0.000182
std,0.009975,0.011255
min,-0.069626,-0.061963
25%,-0.005222,-0.005845
50%,0.000684,0.000206
75%,0.005980,0.006645
max,0.047250,0.074547


# TASK 1

In [59]:
# Mean daily returns
mu = returns.mean()

print("Mean daily returns:")
display(sp.Matrix(mu))

Mean daily returns:


Matrix([
[ 0.00039425543107581],
[0.000182141802285339]])

In [60]:
# Daily covariance matrix
Sigma = returns.cov()

print("Daily covariance matrix:")
display(sp.Matrix(Sigma))

Daily covariance matrix:


Matrix([
[9.95021025814444e-5,  7.73223285477264e-5],
[7.73223285477264e-5, 0.000126665732583136]])

In [61]:
correlation = returns.corr()

print("Correlation matrix:")
display(sp.Matrix(correlation))

Correlation matrix:


Matrix([
[              1.0, 0.688746215727041],
[0.688746215727041,               1.0]])

In [62]:
ff = pd.read_csv("ff_factors_daily.csv", index_col="Date", parse_dates=True)

# Keep only the columns we need
ff = ff[["Mkt-RF", "RF"]]

# Convert from percentage points to decimals
ff = ff / 100

ff.head()

,Mkt-RF,RF
Date,,
2021-01-04,-0.0141,0.0
2021-01-05,0.0086,0.0
2021-01-06,0.0079,0.0
2021-01-07,0.0176,0.0
2021-01-08,0.0051,0.0


In [63]:
data = returns.join(ff, how="inner")

data.head()

,KO,PEP,Mkt-RF,RF
Date,,,,
2021-01-05,-0.010993,0.002980,0.0086,0.0
2021-01-06,-0.031813,-0.012232,0.0079,0.0
2021-01-07,-0.011084,-0.003218,0.0176,0.0
2021-01-08,0.022418,0.012003,0.0051,0.0
2021-01-11,-0.017228,-0.014496,-0.0052,0.0


In [64]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-01-05 to 2025-12-31
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   KO      1254 non-null   float64
 1   PEP     1254 non-null   float64
 2   Mkt-RF  1254 non-null   float64
 3   RF      1254 non-null   float64
dtypes: float64(4)
memory usage: 49.0 KB


In [65]:
data.isna().sum()

KO        0
PEP       0
Mkt-RF    0
RF        0
dtype: int64

In [66]:
data["KO_excess"] = data["KO"] - data["RF"]
data["PEP_excess"] = data["PEP"] - data["RF"]

data.head()

,KO,PEP,Mkt-RF,RF,KO_excess,PEP_excess
Date,,,,,,
2021-01-05,-0.010993,0.002980,0.0086,0.0,-0.010993,0.002980
2021-01-06,-0.031813,-0.012232,0.0079,0.0,-0.031813,-0.012232
2021-01-07,-0.011084,-0.003218,0.0176,0.0,-0.011084,-0.003218
2021-01-08,0.022418,0.012003,0.0051,0.0,0.022418,0.012003
2021-01-11,-0.017228,-0.014496,-0.0052,0.0,-0.017228,-0.014496


In [67]:
# Market factor
X = data["Mkt-RF"]

# Add constant for alpha
X = sm.add_constant(X)

In [68]:
# KO factor regression
ko_model = sm.OLS(data["KO_excess"], X).fit()

print(ko_model.summary())

                            OLS Regression Results                            
Dep. Variable:              KO_excess   R-squared:                       0.096
Model:                            OLS   Adj. R-squared:                  0.096
Method:                 Least Squares   F-statistic:                     133.5
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           2.05e-29
Time:                        22:32:42   Log-Likelihood:                 4062.4
No. Observations:                1254   AIC:                            -8121.
Df Residuals:                    1252   BIC:                            -8111.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0001      0.000      0.530      0.5

In [69]:
# PEP factor regression
pep_model = sm.OLS(data["PEP_excess"], X).fit()

print(pep_model.summary())

                            OLS Regression Results                            
Dep. Variable:             PEP_excess   R-squared:                       0.084
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     114.8
Date:                Tue, 24 Nov 2026   Prob (F-statistic):           1.11e-25
Time:                        22:32:42   Log-Likelihood:                 3902.4
No. Observations:                1254   AIC:                            -7801.
Df Residuals:                    1252   BIC:                            -7791.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -7.653e-05      0.000     -0.251      0.8

In [70]:
B = np.array([
    [ko_model.params["Mkt-RF"]],
    [pep_model.params["Mkt-RF"]]
])
display(Math("B ="))
display(sp.Matrix(B))

<IPython.core.display.Math object>

Matrix([
[0.277462574802436],
[0.292255401234823]])

In [71]:
Omega = np.array([
    [data["Mkt-RF"].var()]
])

display(Math(r"\Omega ="))
display(sp.Matrix(Omega))

<IPython.core.display.Math object>

Matrix([[0.00012459045837677]])

In [72]:
D = np.diag([
    ko_model.resid.var(),
    pep_model.resid.var()
])
display(Math(r"D ="))
display(sp.Matrix(D))

<IPython.core.display.Math object>

Matrix([
[8.99502104559766e-5,                  0.0],
[                0.0, 0.000116100657635109]])

In [73]:
Sigma_factor = B @ Omega @ B.T + D

display(Math(r"\Sigma_{\mathrm{factor}} ="))
display(sp.Matrix(Sigma_factor))

<IPython.core.display.Math object>

Matrix([
[9.95418667493616e-5,  1.01030323117477e-5],
[1.01030323117477e-5, 0.000126742329810395]])

In [74]:
Sigma = returns.cov()

In [75]:
returns.cov().values

array([[9.95021026e-05, 7.73223285e-05],
       [7.73223285e-05, 1.26665733e-04]])

In [76]:
Sigma_sample = returns.cov().values

display(Math(r"\Sigma_{\mathrm{sample}} ="))
display(sp.Matrix(Sigma_sample))

<IPython.core.display.Math object>

Matrix([
[9.95021025814444e-5,  7.73223285477264e-5],
[7.73223285477264e-5, 0.000126665732583136]])

In [77]:
display(Math(r"\Sigma_{\mathrm{sample}} ="))
display(sp.Matrix(Sigma_sample))

display(Math(r"\Sigma_{\mathrm{factor}} = B\Omega B^T + D ="))
display(sp.Matrix(Sigma_factor))

<IPython.core.display.Math object>

Matrix([
[9.95021025814444e-5,  7.73223285477264e-5],
[7.73223285477264e-5, 0.000126665732583136]])

<IPython.core.display.Math object>

Matrix([
[9.95418667493616e-5,  1.01030323117477e-5],
[1.01030323117477e-5, 0.000126742329810395]])

# Task 2

In [78]:
mu_vec= data[["KO_excess", "PEP_excess"]].mean().values

display(Math(r"\mu ="))
display(sp.Matrix(mu_vec))

<IPython.core.display.Math object>

Matrix([
[0.000264111890405953],
[ 5.19982616154826e-5]])

In [79]:
Sigma_sample

array([[9.95021026e-05, 7.73223285e-05],
       [7.73223285e-05, 1.26665733e-04]])

In [80]:
lamda = 4
w_naive = (1 / lamda) * np.linalg.solve(Sigma_sample, mu_vec)

display(Math(r"w_{\mathrm{naive}} ="))
display(sp.Matrix(w_naive))

<IPython.core.display.Math object>

Matrix([
[ 1.11073007332766],
[-0.57540953474511]])

### Nudging $\mu$

In [81]:
# Error-maximizer experiment: shock KO expected return by different basis points

bp_shocks = [0, 1, 2, 3, 5]

results = []

for bp in bp_shocks:
    
    # Nudge KO expected return
    mu_perturbed = mu_vec.copy()
    mu_perturbed[0] += bp / 10000
    
    # Recalculate naive MVO weights
    w_perturbed = (1 / lamda) * np.linalg.solve(
        Sigma_sample,
        mu_perturbed
    )
    
    # Change from original portfolio
    weight_change = w_perturbed - w_naive
    
    results.append({
        "KO Shock (bp)": bp,
        "KO Weight": w_perturbed[0],
        "PEP Weight": w_perturbed[1],
        "KO Weight Change (pp)": weight_change[0] * 100,
        "PEP Weight Change (pp)": weight_change[1] * 100
    })

# Display results
sensitivity_results = pd.DataFrame(results)

sensitivity_results.round(4)

,KO Shock (bp),KO Weight,PEP Weight,KO Weight Change (pp),PEP Weight Change (pp)
0,0,1.1107,-0.5754,0.0000,0.0000
1,1,1.5887,-0.8672,47.8001,-29.1793
2,2,2.0667,-1.1590,95.6002,-58.3586
3,3,2.5447,-1.4508,143.4003,-87.5378
4,5,3.5007,-2.0344,239.0005,-145.8964


# Task 3

In [82]:
# Ridge stabilization with eta values appropriate for daily covariance

eta_values = [0, 0.0001, 0.0005, 0.001, 0.005]

ridge_results = []

for eta in eta_values:
    
    w_ridge = np.linalg.solve(
        lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
        mu_vec
    )
    
    ridge_results.append({
        "Eta": eta,
        "KO Weight": w_ridge[0],
        "PEP Weight": w_ridge[1],
        "Gross Leverage": np.sum(np.abs(w_ridge))
    })

ridge_results = pd.DataFrame(ridge_results)

ridge_results.round(4)

,Eta,KO Weight,PEP Weight,Gross Leverage
0,0.0000,1.1107,-0.5754,1.6861
1,0.0001,0.6982,-0.2702,0.9684
2,0.0005,0.3090,-0.0433,0.3523
3,0.0010,0.1899,-0.0045,0.1944
4,0.0050,0.0485,0.0067,0.0553


In [83]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

# Colour guideline
KO_COLOR, PEP_COLOR = "#E41E2B", "#0057B8"   # Coca-Cola red, Pepsi blue
INK, INK_2, GRID = "#0B0B0B", "#52514E", "#E6E5E1"
LEVERAGE_COLOR = INK

# Data
df_bars = pd.DataFrame({
    "Eta": np.tile(ridge_results["Eta"].astype(str), 2),
    "Series": ["KO Weight"] * len(ridge_results) + ["PEP Weight"] * len(ridge_results),
    "Weight": pd.concat([ridge_results["KO Weight"], ridge_results["PEP Weight"]]).values
})

df_line = pd.DataFrame({
    "Eta": ridge_results["Eta"].astype(str),
    "Gross Leverage": ridge_results["Gross Leverage"]
})

# Scale leverage onto the weight axis so it can share the plot,
# then label each point with its true leverage value.
w_max = df_bars["Weight"].abs().max()
l_max = df_line["Gross Leverage"].max()
scale_factor = (w_max / l_max) * 0.9
df_line["Leverage_scaled"] = df_line["Gross Leverage"] * scale_factor
df_line["Label"] = df_line["Gross Leverage"].round(2).astype(str)

p = (
    ggplot()
    + geom_bar(
        aes(x="Eta", y="Weight", fill="Series"),
        data=df_bars, stat='identity', position=position_dodge(width=0.7),
        width=0.6
    )
    + geom_line(
        aes(x="Eta", y="Leverage_scaled"),
        data=df_line, color=LEVERAGE_COLOR, size=1.2, group=1
    )
    + geom_point(
        aes(x="Eta", y="Leverage_scaled"),
        data=df_line, color=LEVERAGE_COLOR, size=3
    )
    + geom_text(
        aes(x="Eta", y="Leverage_scaled", label="Label"),
        data=df_line, color=LEVERAGE_COLOR, size=8,
        nudge_y=w_max * 0.06
    )
    + scale_fill_manual(values={"KO Weight": KO_COLOR, "PEP Weight": PEP_COLOR})
    + geom_hline(yintercept=0, color=INK_2, size=0.5)
    + labs(x="Eta", y="Weight  (labels = Gross Leverage)",
           title="Ridge-Stabilized Weights and Gross Leverage vs. Eta")
    + theme(
        axis_text=element_text(color=INK_2),
        axis_title=element_text(color=INK),
        plot_title=element_text(color=INK),
        panel_grid_major_y=element_line(color=GRID),
        panel_grid_minor=element_blank(),
        panel_grid_major_x=element_blank(),
        legend_title=element_blank()
    )
    + ggsize(750, 480)
)

p

In [84]:
# Test sensitivity of ridge portfolio to the same KO return shocks

eta = 0.0001
bp_shocks = [0, 1, 2, 3, 5]

ridge_sensitivity = []

# Baseline ridge weights
w_ridge_base = np.linalg.solve(
    lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
    mu_vec
)

for bp in bp_shocks:
    
    # Perturb KO expected return
    mu_perturbed = mu_vec.copy()
    mu_perturbed[0] += bp / 10000
    
    # Ridge portfolio with perturbed mean
    w_ridge = np.linalg.solve(
        lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
        mu_perturbed
    )
    
    # Weight change from baseline ridge portfolio
    weight_change = w_ridge - w_ridge_base
    
    ridge_sensitivity.append({
        "KO Shock (bp)": bp,
        "KO Weight": w_ridge[0],
        "PEP Weight": w_ridge[1],
        "KO Weight Change (pp)": weight_change[0] * 100,
        "PEP Weight Change (pp)": weight_change[1] * 100
    })

ridge_sensitivity = pd.DataFrame(ridge_sensitivity)

ridge_sensitivity.round(4)

,KO Shock (bp),KO Weight,PEP Weight,KO Weight Change (pp),PEP Weight Change (pp)
0,0,0.6982,-0.2702,0.0000,0.0000
1,1,0.9920,-0.4200,29.3836,-14.9803
2,2,1.2858,-0.5698,58.7671,-29.9607
3,3,1.5797,-0.7196,88.1507,-44.9410
4,5,2.1673,-1.0192,146.9178,-74.9017


In [85]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

# Colour guideline
KO_COLOR, PEP_COLOR = "#E41E2B", "#0057B8"   # Coca-Cola red, Pepsi blue
INK, INK_2, GRID = "#0B0B0B", "#52514E", "#E6E5E1"

# Data
df_lines = pd.DataFrame({
    "KO Shock (bp)": np.tile(ridge_sensitivity["KO Shock (bp)"], 2),
    "Series": ["KO Weight Change"] * len(ridge_sensitivity) + ["PEP Weight Change"] * len(ridge_sensitivity),
    "Weight Change (pp)": pd.concat([
        ridge_sensitivity["KO Weight Change (pp)"],
        ridge_sensitivity["PEP Weight Change (pp)"]
    ]).values
})
df_lines["Label"] = df_lines["Weight Change (pp)"].round(1).astype(str)

# Nudge KO labels up and PEP labels down so they don't collide with the lines/points
y_range = df_lines["Weight Change (pp)"].max() - df_lines["Weight Change (pp)"].min()
nudge = y_range * 0.06
df_lines["Label_y"] = np.where(
    df_lines["Series"] == "KO Weight Change",
    df_lines["Weight Change (pp)"] + nudge,
    df_lines["Weight Change (pp)"] - nudge
)

p = (
    ggplot(df_lines, aes(x="KO Shock (bp)", y="Weight Change (pp)", color="Series"))
    + geom_hline(yintercept=0, color=INK_2, size=0.5)
    + geom_line(size=1.2)
    + geom_point(size=3.5)
    + geom_text(
        aes(x="KO Shock (bp)", y="Label_y", label="Label", color="Series"),
        size=8, show_legend=False
    )
    + scale_color_manual(values={"KO Weight Change": KO_COLOR, "PEP Weight Change": PEP_COLOR})
    + scale_x_continuous(breaks=list(ridge_sensitivity["KO Shock (bp)"]))
    + labs(
        x="KO Shock (bp)",
        y="Weight Change (pp)",
        title="Ridge Portfolio Sensitivity to KO Return Shocks",
        color=""
    )
    + theme(
        axis_text=element_text(color=INK_2),
        axis_title=element_text(color=INK),
        plot_title=element_text(color=INK),
        panel_grid_major=element_line(color=GRID),
        panel_grid_minor=element_blank(),
        legend_position="top"
    )
    + ggsize(700, 450)
)

p

In [86]:
# Test sensitivity of ridge portfolio to PEP return shocks

eta = 0.0001
bp_shocks = [0, 1, 2, 3, 5]

ridge_sensitivity_pep = []

# Baseline ridge weights
w_ridge_base = np.linalg.solve(
    lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
    mu_vec
)

for bp in bp_shocks:

    # Perturb PEP expected return
    mu_perturbed = mu_vec.copy()
    mu_perturbed[1] += bp / 10000

    # Ridge portfolio with perturbed mean
    w_ridge = np.linalg.solve(
        lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
        mu_perturbed
    )

    # Weight change from baseline ridge portfolio
    weight_change = w_ridge - w_ridge_base

    ridge_sensitivity_pep.append({
        "PEP Shock (bp)": bp,
        "KO Weight": w_ridge[0],
        "PEP Weight": w_ridge[1],
        "KO Weight Change (pp)": weight_change[0] * 100,
        "PEP Weight Change (pp)": weight_change[1] * 100
    })

ridge_sensitivity_pep = pd.DataFrame(ridge_sensitivity_pep)

ridge_sensitivity_pep.round(4)

,PEP Shock (bp),KO Weight,PEP Weight,KO Weight Change (pp),PEP Weight Change (pp)
0,0,0.6982,-0.2702,0.0000,0.0000
1,1,0.5484,-0.0290,-14.9803,24.1209
2,2,0.3986,0.2122,-29.9607,48.2418
3,3,0.2487,0.4534,-44.9410,72.3627
4,5,-0.0509,0.9358,-74.9017,120.6045


In [87]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

# Colour guideline
KO_COLOR, PEP_COLOR = "#E41E2B", "#0057B8"   # Coca-Cola red, Pepsi blue
INK, INK_2, GRID = "#0B0B0B", "#52514E", "#E6E5E1"

# --- Data (from ridge_sensitivity_pep) ---
df_lines = pd.DataFrame({
    "PEP Shock (bp)": np.tile(ridge_sensitivity_pep["PEP Shock (bp)"], 2),
    "Series": ["KO Weight Change"] * len(ridge_sensitivity_pep) + ["PEP Weight Change"] * len(ridge_sensitivity_pep),
    "Weight Change (pp)": pd.concat([
        ridge_sensitivity_pep["KO Weight Change (pp)"],
        ridge_sensitivity_pep["PEP Weight Change (pp)"]
    ]).values
})
df_lines["Label"] = df_lines["Weight Change (pp)"].round(1).astype(str)

# Nudge PEP labels up and KO labels down so they don't collide with the lines/points
y_range = df_lines["Weight Change (pp)"].max() - df_lines["Weight Change (pp)"].min()
nudge = y_range * 0.06
df_lines["Label_y"] = np.where(
    df_lines["Series"] == "PEP Weight Change",
    df_lines["Weight Change (pp)"] + nudge,
    df_lines["Weight Change (pp)"] - nudge
)

p_pep = (
    ggplot(df_lines, aes(x="PEP Shock (bp)", y="Weight Change (pp)", color="Series"))
    + geom_hline(yintercept=0, color=INK_2, size=0.5)
    + geom_line(size=1.2)
    + geom_point(size=3.5)
    + geom_text(
        aes(x="PEP Shock (bp)", y="Label_y", label="Label", color="Series"),
        size=8, show_legend=False
    )
    + scale_color_manual(values={"KO Weight Change": KO_COLOR, "PEP Weight Change": PEP_COLOR})
    + scale_x_continuous(breaks=list(ridge_sensitivity_pep["PEP Shock (bp)"]))
    + labs(
        x="PEP Shock (bp)",
        y="Weight Change (pp)",
        title="Ridge Portfolio Sensitivity to PEP Return Shocks",
        color=""
    )
    + theme(
        axis_text=element_text(color=INK_2),
        axis_title=element_text(color=INK),
        plot_title=element_text(color=INK),
        panel_grid_major=element_line(color=GRID),
        panel_grid_minor=element_blank(),
        legend_position="top"
    )
    + ggsize(700, 450)
)

p_pep

In [88]:
# Compare naive MVO vs ridge sensitivity

comparison = []

for bp in [1, 2, 3, 5]:
    
    mu_shock = mu_vec.copy()
    mu_shock[0] += bp / 10000

    # Naive MVO
    w_naive_shock = (1 / lamda) * np.linalg.solve(
        Sigma_sample, mu_shock
    )
    naive_change = w_naive_shock - w_naive
    naive_turnover = np.sum(np.abs(naive_change))

    # Ridge MVO
    w_ridge_shock = np.linalg.solve(
        lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
        mu_shock
    )
    ridge_change = w_ridge_shock - w_ridge_base
    ridge_turnover = np.sum(np.abs(ridge_change))

    comparison.append({
        "KO Shock (bp)": bp,
        "Naive Turnover": naive_turnover,
        "Ridge Turnover": ridge_turnover,
        "Turnover Reduction (%)":
            (1 - ridge_turnover / naive_turnover) * 100
    })

comparison = pd.DataFrame(comparison)

comparison.round(4)

,KO Shock (bp),Naive Turnover,Ridge Turnover,Turnover Reduction (%)
0,1,0.7698,0.4436,42.3691
1,2,1.5396,0.8873,42.3691
2,3,2.3094,1.3309,42.3691
3,5,3.8490,2.2182,42.3691


In [89]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

# Colour guideline
KO_COLOR, PEP_COLOR = "#E41E2B", "#0057B8"   # Coca-Cola red, Pepsi blue
INK, INK_2, GRID = "#0B0B0B", "#52514E", "#E6E5E1"

# Naive vs Ridge comparison colors
NAIVE_COLOR, RIDGE_COLOR = "#E97131", "#1B9AAA"

# --- Data (from comparison) ---
df_bars = pd.DataFrame({
    "KO Shock (bp)": np.tile(comparison["KO Shock (bp)"].astype(str), 2),
    "Method": ["Naive"] * len(comparison) + ["Ridge"] * len(comparison),
    "Turnover": pd.concat([
        comparison["Naive Turnover"],
        comparison["Ridge Turnover"]
    ]).values
})
df_bars["Label"] = df_bars["Turnover"].round(2).astype(str)

avg_reduction = comparison["Turnover Reduction (%)"].mean()

p_turnover = (
    ggplot(df_bars, aes(x="KO Shock (bp)", y="Turnover", fill="Method"))
    + geom_bar(stat='identity', position=position_dodge(width=0.7), width=0.6)
    + geom_label(
        aes(label="Label"),
        position=position_dodge(width=0.7),
        va='bottom', size=8, color=INK, fill='white',
        label_size=0, label_padding=0.15
    )
    + scale_fill_manual(values={"Naive": NAIVE_COLOR, "Ridge": RIDGE_COLOR})
    + labs(
        x="KO Shock (bp)",
        y="Turnover",
        title="Naive vs Ridge Portfolio Turnover Under KO Return Shocks",
        subtitle=f"Ridge reduces turnover by ~{avg_reduction:.1f}% at every shock level",
        fill=""
    )
    + theme(
        axis_text=element_text(color=INK_2),
        axis_title=element_text(color=INK),
        plot_title=element_text(color=INK),
        plot_subtitle=element_text(color=INK_2),
        panel_grid_major_y=element_line(color=GRID),
        panel_grid_minor=element_blank(),
        panel_grid_major_x=element_blank(),
        legend_position="top"
    )
    + ggsize(700, 450)
)

p_turnover

In [90]:
# Compare naive MVO vs ridge sensitivity — PEP shocks

comparison_pep = []

for bp in [1, 2, 3, 5]:
    
    mu_shock = mu_vec.copy()
    mu_shock[1] += bp / 10000

    # Naive MVO
    w_naive_shock = (1 / lamda) * np.linalg.solve(
        Sigma_sample, mu_shock
    )
    naive_change = w_naive_shock - w_naive
    naive_turnover = np.sum(np.abs(naive_change))

    # Ridge MVO
    w_ridge_shock = np.linalg.solve(
        lamda * Sigma_sample + eta * np.eye(len(mu_vec)),
        mu_shock
    )
    ridge_change = w_ridge_shock - w_ridge_base
    ridge_turnover = np.sum(np.abs(ridge_change))

    comparison_pep.append({
        "PEP Shock (bp)": bp,
        "Naive Turnover": naive_turnover,
        "Ridge Turnover": ridge_turnover,
        "Turnover Reduction (%)":
            (1 - ridge_turnover / naive_turnover) * 100
    })

comparison_pep = pd.DataFrame(comparison_pep)

comparison_pep.round(4)

,PEP Shock (bp),Naive Turnover,Ridge Turnover,Turnover Reduction (%)
0,1,0.6673,0.3910,41.4025
1,2,1.3346,0.7820,41.4025
2,3,2.0019,1.1730,41.4025
3,5,3.3364,1.9551,41.4025


In [91]:
import numpy as np
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

# Colour guideline
KO_COLOR, PEP_COLOR = "#E41E2B", "#0057B8"   # Coca-Cola red, Pepsi blue
INK, INK_2, GRID = "#0B0B0B", "#52514E", "#E6E5E1"

# Naive vs Ridge comparison colors
NAIVE_COLOR, RIDGE_COLOR = "#E97131", "#1B9AAA"

# --- Data (from comparison_pep) ---
df_bars = pd.DataFrame({
    "PEP Shock (bp)": np.tile(comparison_pep["PEP Shock (bp)"].astype(str), 2),
    "Method": ["Naive"] * len(comparison_pep) + ["Ridge"] * len(comparison_pep),
    "Turnover": pd.concat([
        comparison_pep["Naive Turnover"],
        comparison_pep["Ridge Turnover"]
    ]).values
})
df_bars["Label"] = df_bars["Turnover"].round(2).astype(str)

avg_reduction = comparison_pep["Turnover Reduction (%)"].mean()

p_turnover_pep = (
    ggplot(df_bars, aes(x="PEP Shock (bp)", y="Turnover", fill="Method"))
    + geom_bar(stat='identity', position=position_dodge(width=0.7), width=0.6)
    + geom_label(
        aes(label="Label"),
        position=position_dodge(width=0.7),
        va='bottom', size=8, color=INK, fill='white',
        label_size=0, label_padding=0.15
    )
    + scale_fill_manual(values={"Naive": NAIVE_COLOR, "Ridge": RIDGE_COLOR})
    + labs(
        x="PEP Shock (bp)",
        y="Turnover",
        title="Naive vs Ridge Portfolio Turnover Under PEP Return Shocks",
        subtitle=f"Ridge reduces turnover by ~{avg_reduction:.1f}% at every shock level",
        fill=""
    )
    + theme(
        axis_text=element_text(color=INK_2),
        axis_title=element_text(color=INK),
        plot_title=element_text(color=INK),
        plot_subtitle=element_text(color=INK_2),
        panel_grid_major_y=element_line(color=GRID),
        panel_grid_minor=element_blank(),
        panel_grid_major_x=element_blank(),
        legend_position="top"
    )
    + ggsize(700, 450)
)

p_turnover_pep

# Task 4

In [92]:
alpha = np.array([
    ko_model.params["const"],
    pep_model.params["const"]
])

display(Math(r"\alpha ="))
display(sp.Matrix(alpha))

<IPython.core.display.Math object>

Matrix([
[0.000142085885618438],
[-7.65335197090827e-5]])

In [93]:
D_inv = np.linalg.inv(D)

w0 = (1 / lamda) * D_inv @ alpha

display(Math(r"w_0 ="))
display(sp.Matrix(w0))

<IPython.core.display.Math object>

Matrix([
[ 0.394901481881407],
[-0.164799927209755]])

In [94]:
scale = (B.T @ w0) / (B.T @ D_inv @ B)

w_factor_neutral = w0 - (D_inv @ B * scale).flatten()

display(Math(r"w^*_{\mathrm{factor\ neutral}} ="))
display(sp.Matrix(w_factor_neutral))

<IPython.core.display.Math object>

Matrix([
[ 0.275887566696639],
[-0.261923216091814]])

In [95]:
factor_exposure = B.T @ w_factor_neutral

display(Math(r"B^\top w^* ="))
display(sp.Matrix(factor_exposure))

<IPython.core.display.Math object>

Matrix([[8.34604393112476e-18]])

In [96]:
# Risk comparison across the three portfolios

portfolios = {
    "Naive MVO": w_naive,
    "Ridge MVO": w_ridge_base,
    "Factor-Neutral": w_factor_neutral
}

te_results = []

for name, w in portfolios.items():
    
    # Benchmark is zero, so portfolio risk is plain volatility
    a = w
    
    # Variance and volatility (sample covariance matrix)
    TE_squared = a.T @ Sigma_sample @ a
    TE_daily = np.sqrt(TE_squared)
    TE_annual = TE_daily * np.sqrt(252)
    
    te_results.append({
        "Portfolio": name,
        "Variance (Daily)": TE_squared,
        "Volatility (Daily)": TE_daily,
        "Volatility (Annualized)": TE_annual
    })

te_results = pd.DataFrame(te_results)

te_results.round(6)

,Portfolio,Variance (Daily),Volatility (Daily),Volatility (Annualized)
0,Naive MVO,0.000066,0.008115,0.128827
1,Ridge MVO,0.000029,0.005345,0.084857
2,Factor-Neutral,0.000005,0.002256,0.035809
